# Clustering des Decks par Style de Jeu (TOK-15)

Classifier automatiquement les decks en **Combo / Control / Midrange / OTK** via clustering.

**Méthode :**
- Features : ratios monster/spell/trap, densité de tags mécaniques (search, negate, quick_effect...), taille extra deck, hand traps
- K-Means k=4 + silhouette pour valider le nombre de clusters
- Labelling des clusters via centroïdes
- Table produite : `deck_style_clusters` (deck_id, style, confidence, cluster)

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

con = sqlite3.connect('../data/yugioh.db')

# ── Chargement des données ─────────────────────────────────────────────────────
deck_cards = pd.read_sql("""
    SELECT dc.deck_id, dc.card_name, dc.amount, dc.zone,
           td.archetype, td.placement
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0
""", con)

card_info = pd.read_sql("""
    SELECT name, type, atk, def, level
    FROM cards
""", con)

tags_df = pd.read_sql("""
    SELECT card_name, tags FROM card_mechanic_tags WHERE tags != ''
""", con)

# Explode tags → une ligne par (card, tag)
tags_df['tag_list'] = tags_df['tags'].str.split(',')
tags_exploded = tags_df.explode('tag_list')
tags_exploded['tag_list'] = tags_exploded['tag_list'].str.strip()
tags_exploded = tags_exploded[tags_exploded['tag_list'] != '']

# Tag binaire par carte (pivot card_name → tag → 1/0)
card_tags = tags_exploded.pivot_table(
    index='card_name', columns='tag_list', values='tag_list',
    aggfunc='count', fill_value=0
).clip(upper=1)

print(f'deck_cards : {len(deck_cards):,} lignes, {deck_cards["deck_id"].nunique():,} decks')
print(f'card_info  : {len(card_info):,} cartes')
print(f'card_tags  : {card_tags.shape[0]:,} cartes × {card_tags.shape[1]} tags')

deck_cards : 154,541 lignes, 3,601 decks
card_info  : 13,797 cartes
card_tags  : 12,217 cartes × 25 tags


## 1. Feature Engineering par Deck

In [2]:
# ── Cartes connues par catégorie ──────────────────────────────────────────────
HAND_TRAPS = {
    'Ash Blossom & Joyous Spring', 'Effect Veiler', 'Maxx "C"',
    'Nibiru, the Primal Being', 'Ghost Ogre & Snow Rabbit',
    'Droll & Lock Bird', 'Ghost Belle & Haunted Mansion',
    'D.D. Crow', 'Skull Meister', 'Retaliating "C"'
}
GOING_SECOND = {
    "Lightning Storm", "Harpie's Feather Duster",
    "Triple Tactics Talent", "Forbidden Droplet",
    "Dark Ruler No More", "Called by the Grave",
    "Lava Golem", "Santa Claws"
}
COUNTER_TRAPS = {
    'Solemn Strike', 'Solemn Warning', 'Solemn Judgment',
    'Solemn Scolding', 'Qi Jia the Impervious Inquisitor'
}

# ── Main deck : ratios de types ────────────────────────────────────────────────
main = deck_cards[deck_cards['zone'] == 'main'].copy()
main = main.merge(card_info.rename(columns={'name': 'card_name'}),
                  on='card_name', how='left')

# Catégorie simplifiée
def simplify_type(t):
    if pd.isna(t): return 'unknown'
    t = str(t)
    if 'Spell' in t: return 'spell'
    if 'Trap' in t:  return 'trap'
    return 'monster'

main['card_cat'] = main['type'].apply(simplify_type)

# Totaux par zone par deck
total_main = main.groupby('deck_id')['amount'].sum().rename('total_main')
type_counts = (main.groupby(['deck_id', 'card_cat'])['amount']
               .sum().unstack(fill_value=0))
for col in ['monster', 'spell', 'trap']:
    if col not in type_counts: type_counts[col] = 0

feats = pd.DataFrame(index=total_main.index)
feats['monster_ratio'] = type_counts['monster'] / total_main
feats['spell_ratio']   = type_counts['spell']   / total_main
feats['trap_ratio']    = type_counts['trap']     / total_main

# Tuner ratio (Synchro indicator)
tuner_cnt = (main[main['type'].str.contains('Tuner', na=False)]
             .groupby('deck_id')['amount'].sum())
feats['tuner_ratio'] = (tuner_cnt / total_main).fillna(0)

# Hand traps
ht_cnt = (main[main['card_name'].isin(HAND_TRAPS)]
          .groupby('deck_id')['amount'].sum())
feats['hand_trap_cnt'] = ht_cnt.fillna(0)

# Going 2nd cards
g2_cnt = (main[main['card_name'].isin(GOING_SECOND)]
          .groupby('deck_id')['amount'].sum())
feats['going_second_cnt'] = g2_cnt.fillna(0)

# Counter traps
ct_cnt = (main[main['card_name'].isin(COUNTER_TRAPS)]
          .groupby('deck_id')['amount'].sum())
feats['counter_trap_cnt'] = ct_cnt.fillna(0)

# ── Tags mécaniques agrégés par deck ──────────────────────────────────────────
main_tags = main.merge(card_tags.reset_index().rename(columns={'card_name': 'card_name'}),
                       on='card_name', how='left')

SIGNAL_TAGS = ['search_deck', 'special_summon', 'negate', 'quick_effect',
                'draw', 'destroy', 'gy_recursion', 'banish', 'counter',
                'bounce', 'opponent_hand']

for tag in SIGNAL_TAGS:
    if tag in main_tags.columns:
        tag_cnt = main_tags.groupby('deck_id').apply(
            lambda g: (g[tag] * g['amount']).sum()
        )
        feats[f'tag_{tag}'] = (tag_cnt / total_main).fillna(0)
    else:
        feats[f'tag_{tag}'] = 0.0

# ── Extra deck size ────────────────────────────────────────────────────────────
extra_size = (deck_cards[deck_cards['zone'] == 'extra']
              .groupby('deck_id')['amount'].sum()
              .rename('extra_size'))
feats['extra_size'] = extra_size.fillna(0)

# ── Merge archetype pour validation ───────────────────────────────────────────
arch_map = deck_cards[['deck_id', 'archetype']].drop_duplicates('deck_id')
feats = feats.merge(arch_map, left_index=True, right_on='deck_id', how='left')
feats = feats.set_index('deck_id')

FEAT_COLS = [c for c in feats.columns if c != 'archetype']

feats_clean = feats[FEAT_COLS].fillna(0)
# Exclure decks avec < 30 cartes main (incomplets)
valid = total_main[total_main >= 30].index
feats_clean = feats_clean.loc[feats_clean.index.isin(valid)]
feats_arch = feats.loc[feats_clean.index, 'archetype']

print(f'Feature matrix : {feats_clean.shape[0]:,} decks × {feats_clean.shape[1]} features')
print(f'Features : {FEAT_COLS}')

Feature matrix : 3,601 decks × 19 features
Features : ['monster_ratio', 'spell_ratio', 'trap_ratio', 'tuner_ratio', 'hand_trap_cnt', 'going_second_cnt', 'counter_trap_cnt', 'tag_search_deck', 'tag_special_summon', 'tag_negate', 'tag_quick_effect', 'tag_draw', 'tag_destroy', 'tag_gy_recursion', 'tag_banish', 'tag_counter', 'tag_bounce', 'tag_opponent_hand', 'extra_size']


## 2. K-Means Clustering (k=4)

In [3]:
# Normalisation
scaler = StandardScaler()
X = scaler.fit_transform(feats_clean)

# Silhouette score pour k=2..6
print('Silhouette scores par k :')
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels, sample_size=5000, random_state=42)
    print(f'  k={k}  silhouette={sil:.3f}')

# Modèle final k=4
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=20, max_iter=500)
labels = km.fit_predict(X)

feats_clean = feats_clean.copy()
feats_clean['cluster'] = labels
print(f'\nDistribution des clusters :')
print(feats_clean['cluster'].value_counts().sort_index())

Silhouette scores par k :
  k=2  silhouette=0.177


  k=3  silhouette=0.164
  k=4  silhouette=0.171


  k=5  silhouette=0.173
  k=6  silhouette=0.207



Distribution des clusters :
cluster
0    1062
1     681
2    1834
3      24
Name: count, dtype: int64


## 3. Labelling automatique des clusters

In [4]:
# Centroïdes en espace original (non-normalisé)
centroids = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=FEAT_COLS
)

print('=== Centroïdes par cluster ===\n')
for k in range(K):
    print(f'--- Cluster {k} (n={sum(labels==k):,}) ---')
    top = centroids.iloc[k].sort_values(ascending=False)
    for feat, val in top.head(8).items():
        print(f'  {feat:30s}: {val:.3f}')
    print()

# Règles de labelling basées sur les features discriminantes
def label_cluster(row):
    """Attribuer un style en fonction du centroïde."""
    # Combo : haute densité de special_summon + search + gros extra deck
    combo_score = row['tag_special_summon'] + row['tag_search_deck'] + row['extra_size']/15
    # Control : haute densité trap + negate + quick_effect
    control_score = row['trap_ratio']*3 + row['tag_negate'] + row['tag_quick_effect'] + row['counter_trap_cnt']/3
    # OTK : special_summon élevé + peu de traps + going_second
    otk_score = row['tag_special_summon'] + row['going_second_cnt'] - row['trap_ratio']*2
    # Midrange : équilibré, beaucoup de hand traps, hand advantage
    midrange_score = row['hand_trap_cnt']/3 + row['tag_draw'] + row['tag_gy_recursion']

    scores = {
        'Combo':    combo_score,
        'Control':  control_score,
        'OTK':      otk_score,
        'Midrange': midrange_score,
    }
    return max(scores, key=scores.get)

cluster_labels = {k: label_cluster(centroids.iloc[k]) for k in range(K)}
print('=== Labels assignés ===')
for k, label in cluster_labels.items():
    n = sum(labels == k)
    print(f'  Cluster {k} → {label:10s}  (n={n:,}, {100*n/len(labels):.1f}%)')

=== Centroïdes par cluster ===

--- Cluster 0 (n=1,062) ---
  extra_size                    : 15.018
  hand_trap_cnt                 : 3.061
  going_second_cnt              : 2.760
  tag_special_summon            : 0.439
  spell_ratio                   : 0.417
  tag_search_deck               : 0.400
  monster_ratio                 : 0.364
  tag_quick_effect              : 0.182

--- Cluster 1 (n=681) ---
  extra_size                    : 15.181
  hand_trap_cnt                 : 9.029
  going_second_cnt              : 0.742
  monster_ratio                 : 0.727
  tag_special_summon            : 0.726
  tag_search_deck               : 0.535
  tuner_ratio                   : 0.511
  tag_quick_effect              : 0.431

--- Cluster 2 (n=1,834) ---
  extra_size                    : 15.083
  hand_trap_cnt                 : 5.533
  going_second_cnt              : 1.479
  tag_special_summon            : 0.655
  monster_ratio                 : 0.611
  tag_search_deck               : 0.552
 

## 4. Validation par archetype

In [5]:
feats_clean['style']    = feats_clean['cluster'].map(cluster_labels)
feats_clean['archetype'] = feats_arch

# Distribution style par archetype (archetypes avec >= 20 decks)
arch_counts = feats_clean['archetype'].value_counts()
big_archs = arch_counts[arch_counts >= 20].index

arch_style = (feats_clean[feats_clean['archetype'].isin(big_archs)]
              .groupby(['archetype', 'style']).size()
              .unstack(fill_value=0))
arch_style['total'] = arch_style.sum(axis=1)
arch_style['dominant_style'] = arch_style.drop('total', axis=1).idxmax(axis=1)
arch_style['pct_dominant'] = arch_style.apply(
    lambda r: r[r['dominant_style']] / r['total'] if r['total'] > 0 else 0, axis=1
)

print('=== Style dominant par archetype (>= 20 decks) ===\n')
print(arch_style[['total','dominant_style','pct_dominant']]
      .sort_values('pct_dominant', ascending=False)
      .to_string(float_format='{:.0%}'.format))

print('\n=== Distribution globale des styles ===')
print(feats_clean['style'].value_counts())

=== Style dominant par archetype (>= 20 decks) ===

style                  total dominant_style  pct_dominant
archetype                                                
Orcust                    37          Combo          100%
Blitzclique               24            OTK          100%
Crystron                  28          Combo          100%
Rokket                    27          Combo          100%
Sky Striker              191            OTK          100%
Kewl Tune                457       Midrange           99%
Vanquish Soul K9         103          Combo           99%
Blue-Eyes                 97          Combo           98%
Magnet Warrior            40          Combo           98%
Dracotail                230          Combo           97%
Centur-Ion                27          Combo           96%
Mermail Atlantean         26          Combo           96%
Labrynth                  48          Combo           96%
Yummy                    154          Combo           95%
Chaos Ritual        

## 5. Confidence score + Sauvegarde

In [6]:
# Confidence = 1 - (distance au centroïde / max distance dans le cluster)
distances = km.transform(X)  # shape: (n_decks, k)
assigned_dist = distances[np.arange(len(labels)), labels]
max_dist_per_cluster = np.array([
    assigned_dist[labels == k].max() if (labels == k).any() else 1
    for k in range(K)
])
confidence = 1 - assigned_dist / max_dist_per_cluster[labels]

out = pd.DataFrame({
    'deck_id':    feats_clean.index,
    'cluster':    labels,
    'style':      feats_clean['style'].values,
    'confidence': confidence.round(3),
    'archetype':  feats_clean['archetype'].values,
})

# Sauvegarder
con2 = sqlite3.connect('../data/yugioh.db')
con2.execute("DROP TABLE IF EXISTS deck_style_clusters")
con2.execute("""
    CREATE TABLE deck_style_clusters (
        deck_id    TEXT PRIMARY KEY,
        cluster    INTEGER,
        style      TEXT,
        confidence REAL,
        archetype  TEXT
    )
""")
out.to_sql('deck_style_clusters', con2, if_exists='append', index=False)
con2.commit()
con2.close()

print(f'✓ {len(out):,} decks sauvegardés dans deck_style_clusters')
print(f'\nDistribution finale :')
print(out['style'].value_counts())
print(f'\nConfidence moyenne : {out["confidence"].mean():.3f}')
print(f'Confidence médiane : {out["confidence"].median():.3f}')

print('\n=== Sample par style ===')
for style in out['style'].unique():
    sub = out[out['style'] == style]
    top_arch = sub['archetype'].value_counts().head(3).index.tolist()
    print(f'  {style:10s}: top archetypes = {top_arch}')

✓ 3,601 decks sauvegardés dans deck_style_clusters

Distribution finale :
style
Combo       1834
OTK         1062
Midrange     681
Control       24
Name: count, dtype: int64

Confidence moyenne : 0.739
Confidence médiane : 0.761

=== Sample par style ===
  Combo     : top archetypes = ['Dracotail', 'Branded', 'Yummy']
  OTK       : top archetypes = ['Sky Striker', 'Radiant Typhoon', 'Maliss']
  Control   : top archetypes = ['Odion', 'Runick', 'Labrynth']
  Midrange  : top archetypes = ['Kewl Tune', 'Elfnote', 'Enneacraft']
